In [ ]:
from datetime import datetime

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm

### 1.  Выбор данных, полученных во время эксперимента

In [ ]:
def get_data_subset(df, begin_date=None, end_date=None, user_ids=None, columns=None):
    """Возвращает подмножество данных.

    :param df (pd.DataFrame): таблица с данными, обязательные столбцы: 'date', 'user_id'.
    :param begin_date (datetime.datetime | None): дата начала интервала с данными.
        Пример, df[df['date'] >= begin_date].
        Если None, то фильтровать не нужно.
    :param end_date (datetime.datetime | None): дата окончания интервала с данными.
        Пример, df[df['date'] < end_date].
        Если None, то фильтровать не нужно.
    :param user_ids (list[str] | None): список user_id, по которым нужно предоставить данные.
        Пример, df[df['user_id'].isin(user_ids)].
        Если None, то фильтровать по user_id не нужно.
    :param columns (list[str] | None): список названий столбцов, по которым нужно предоставить данные.
        Пример, df[columns].
        Если None, то фильтровать по columns не нужно.

    :return df (pd.DataFrame): датафрейм с подмножеством данных.
    """
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df.date.min()
    if user_ids and columns:
        return df[(df.date >= begin_date) & (df.date < end_date) & (df.user_id.isin(user_ids))][columns]
    elif user_ids:
        return df[(df.date >= begin_date) & (df.date < end_date) & (df.user_id.isin(user_ids))]
    elif columns:
        return df[(df.date >= begin_date) & (df.date < end_date)][columns]
    else:
        return df[(df.date >= begin_date) & (df.date < end_date)]

In [ ]:
df = pd.DataFrame({
    'date': [datetime(2022, 1, 5), datetime(2022, 1, 7)],
    'user_id': ['1', '2'],
})
new_df = get_data_subset(df, datetime(2022, 1, 1), datetime(2022, 1, 6))

### 2. Время обработки запроса сервером

In [ ]:
def get_response_time(df_web_logs, begin_date, end_date):
    """Вычисляет значения времени обработки запроса сервером.

    Нужно вернуть значения user_id и load_time из таблицы df_web_logs,
    отфильтрованные по дате.
    Считаем, что запросы обрабатываются независимо, поэтому группировать
    по user_id не нужно.

    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
    столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для
    фильтрации данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """

    
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
    result = df_web_logs[(df_web_logs.date >= begin_date) & (df_web_logs.date < end_date)][['user_id', 'load_time']]
    result.columns= ['user_id', 'metric']
    return result

### 3. Выручка с пользователя за указанный период

In [ ]:
def get_revenue_web(df_sales, df_web_logs, begin_date, end_date):
    """Вычисляет значения выручки с пользователя за указанный период
    для заходивших на сайт в указанный период.

    Эти данные нужны для экспериментов на сайте, когда в эксперимент
    попадают только те, кто заходил на сайт во время эксперимента.

    Нужно вернуть значения user_id и выручки (sum(price)) за указанный
    период для пользователей, заходивших на сайт в указанный период.
    Если пользователь зашёл на сайт и ничего не купил, его суммарная
    стоимость покупок равна нулю.
    Для каждого user_id должно быть ровно одно значение.

    :param df_sales (pd.DataFrame): таблица с продажами, содержит
        столбцы ['user_id', 'date', 'price'].
    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
        столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для фильтрации
        данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """

    
    # handle cases with None initial parameters
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
        
    # get users, visited site in period under investigation
    users = df_web_logs[(df_web_logs.date >= begin_date) & (df_web_logs.date < end_date)][['user_id']].drop_duplicates()
    
    # get sales in period under investigation
    sales = df_sales[
                        (df_sales.date >= begin_date) 
                        & (df_sales.date < end_date) 
                        & (df_sales['user_id'].isin(users.user_id.to_list()))
                    ][['user_id', 'price']]
    sales = sales.groupby(['user_id'], as_index=False).agg(metric = ('price', 'sum'))
    
    # merge dataframes
    return users.merge(sales, on=['user_id'], how='left').fillna(0)

### 4. Значения выручки с пользователя за указанный период для заходивших на сайт до end_date

In [ ]:
def get_revenue_all(df_sales, df_web_logs, begin_date, end_date):
    """Вычисляет значения выручки с пользователя за указанный период
    для заходивших на сайт до end_date.

    Эти данные нужны, например, для экспериментов с рассылкой по email,
    когда в эксперимент попадают те, кто когда-либо оставил нам свои данные.

    Нужно вернуть значения user_id и выручки (sum(price)) за указанный период
    для пользователей, заходивших на сайт до end_date.
    Если пользователь ничего не купил за указанный период, его суммарная
    стоимость покупок равна нулю.
    Для каждого user_id должно быть ровно одно значение.

    :param df_sales (pd.DataFrame): таблица с продажами, содержит
        столбцы ['user_id', 'date', 'price'].
    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
        столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для фильтрации
        данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """

    
    # handle cases with None initial parameters
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
        
    # get users, visited site before end_date
    users = df_web_logs[(df_web_logs.date < end_date)][['user_id']].drop_duplicates()
    
    # get sales in period under investigation
    sales = df_sales[
                        (df_sales.date >= begin_date) 
                        & (df_sales.date < end_date) 
                        & (df_sales['user_id'].isin(users.user_id.to_list()))
                    ][['user_id', 'price']]
    sales = sales.groupby(['user_id'], as_index=False).agg(metric = ('price', 'sum'))
    
    # merge dataframes
    return users.merge(sales, on=['user_id'], how='left').fillna(0)

### 5. Расчет p-value

In [ ]:
def get_ttest_pvalue(metrics_a_group, metrics_b_group):
    """Применяет тест Стьюдента, возвращает pvalue.

    :param metrics_a_group (np.array): массив значений метрик группы A
    :param metrics_a_group (np.array): массив значений метрик группы B
    :return (float): значение p-value
    """

    
    return stats.ttest_ind(metrics_a_group, metrics_b_group).pvalue.item()

### 6. Расчет необходимого размера выборки

MDE из доступного объема /2 (если 2 варианта)

In [ ]:
def get_minimal_determinable_effect(std, sample_size, alpha=0.05, beta=0.2):
    t_alpha = norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = norm.ppf(1 - beta, loc=0, scale=1)
    disp_sum_sqrt = (2 * (std ** 2)) ** 0.5
    mde = (t_alpha + t_beta) * disp_sum_sqrt / np.sqrt(sample_size)
    return mde

Расчет размера выборки

In [ ]:
def estimate_sample_size(metrics, effect, alpha, beta):
    """Оцениваем необходимый размер выборки для проверки гипотезы о равенстве средних.
    
    Для метрик, у которых для одного пользователя одно значение просто вычислите
    размер групп по формуле.
    Для метрик, у которых для одного пользователя несколько значений (например,
    response_time), вычислите необходимый объём данных и разделите его на среднее
    количество значений на одного пользователя.
    Пример, если в таблице metrics 1000 наблюдений и 100 уникальных пользователей,
    и для эксперимента нужно 302 наблюдения, то размер групп будет 31, тк в среднем на
    одного пользователя 10 наблюдений, то получится порядка 310 наблюдений в группе.

    :param metrics (pd.DataFrame): таблица со значениями метрик,
        содержит столбцы ['user_id', 'metric'].
    :param effect (float): размер эффекта в процентах.
        Пример, effect=3 означает, что ожидаем увеличение среднего на 3%.
    :param alpha (float): уровень значимости.
    :param beta (float): допустимая вероятность ошибки II рода.
    :return (int): минимально необходимый размер групп (количество пользователей), т.е. размер 
        контрольной или экспериментальной группы.
    """


    # general count of observations
    t_alpha = norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = norm.ppf(1 - beta, loc=0, scale=1)
    z_scores_sum_squared = (t_alpha + t_beta) ** 2
    epsilon = (effect / 100) * metrics.metric.mean()
    std = np.std(metrics.metric)
    sample_size = int(
        np.ceil(
            z_scores_sum_squared * (2 * std ** 2) / (epsilon ** 2)
        )
    )
    
    # observations count depending of metric's type
    metrics_type = metrics.groupby(['user_id']).agg(metric_count=('metric', 'count'))     
    if metrics_type.metric_count.max() == 1:
        return round(sample_size, 0)
    else:
        observation_per_user = metrics.shape[0] / metrics.user_id.nunique()
        return int(np.ceil(sample_size / observation_per_user))

In [ ]:
# validation
metrics = pd.DataFrame({
    'user_id': np.arange(100),
    'metric': np.linspace(500, 1490, 100)
})
effect, alpha, beta = 3, 0.05, 0.1
sample_size = estimate_sample_size(metrics, effect, alpha, beta)
# sample_size = 1966
sample_size

### 7. Вероятности ошибок I и II рода

Генерируем выборки из последовательности

In [ ]:
def create_group_generator(metrics, sample_size, n_iter):
    """Генератор случайных групп.

    :param metrics (pd.DataFame): таблица с метриками, columns=['user_id', 'metric'].
    :param sample_size (int): размер групп (количество пользователей в группе).
    :param n_iter (int): количество итераций генерирования случайных групп.
    :return (np.array, np.array): два массива со значениями метрик в группах.
    """
    user_ids = metrics['user_id'].unique()
    for _ in range(n_iter):
        a_user_ids, b_user_ids = np.random.choice(user_ids, (2, sample_size), False)
        a_metric_values = metrics.loc[metrics['user_id'].isin(a_user_ids), 'metric'].values
        b_metric_values = metrics.loc[metrics['user_id'].isin(b_user_ids), 'metric'].values
        yield a_metric_values, b_metric_values

metrics = pd.DataFrame({'user_id': [1, 2, 3, 4], 'metric': [5, 6, 8, 9.1] })
sample_size = 2
n_iter = 3
group_generator = create_group_generator(metrics, sample_size, n_iter)

Оценка вероятности ошибок

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


def estimate_errors(group_generator, effect_add_type, effect, alpha):
    """Оцениваем вероятности ошибок I и II рода.

    :param group_generator: генератор значений метрик для двух групп.
    :param effect_add_type (str): способ добавления эффекта для группы B.
        - 'all_const' - увеличить всем значениям в группе B на константу (b_metric_values.mean() * effect / 100).
        - 'all_percent' - увеличить всем значениям в группе B в (1 + effect / 100) раз.
    :param effect (float): размер эффекта в процентах.
        Пример, effect=3 означает, что ожидаем увеличение среднего на 3%.
    :param alpha (float): уровень значимости.
    :return pvalues_aa (list[float]), pvalues_ab (list[float]), first_type_error (float), second_type_error (float):
        - pvalues_aa, pvalues_ab - списки со значениями pvalue
        - first_type_error, second_type_error - оценки вероятностей ошибок I и II рода.
    """
    pvalues_aa = []
    pvalues_ab = []
    aa_errors = 0
    ab_errors = 0
    aa_tests_count = 0
    ab_tests_count = 0

    for metrics_a_group, metrics_b_group in group_generator:
        #A/A test
        t_stat_aa, p_value_aa = stats.ttest_ind(metrics_a_group, metrics_b_group)
        pvalues_aa.append(p_value_aa)
        if p_value_aa < alpha:
            aa_errors += 1
        aa_tests_count += 1
        
        #Add effect to metric b
        if effect_add_type == 'all_const':
            effect_value = metrics_b_group.mean() * effect / 100
            metrics_b_group += effect_value
        elif effect_add_type == 'all_percent':
            metrics_b_group *= 1 + effect / 100

        # A/B test
        t_stat_ab, p_value_ab = stats.ttest_ind(metrics_a_group, metrics_b_group)
        pvalues_ab.append(p_value_ab)
        if p_value_ab >= alpha:
            ab_errors += 1
        ab_tests_count += 1

        #Type I error: false positive
        typeI_error = aa_errors / aa_tests_count

        #Type II error: false negative
        typeII_error = ab_errors / ab_tests_count

    return pvalues_aa, pvalues_ab, typeI_error, typeII_error

In [ ]:
# validation
sample_size, n_iter, effect, alpha = 100, 10, 6, 0.05

group_generator = (
    (np.arange(sample_size, dtype=float), np.arange(sample_size, dtype=float) + x,)
    for x in range(n_iter)
)
effect_add_type = 'all_const'
pvalues_aa, pvalues_ab, first_type_error, second_type_error = estimate_errors(
    group_generator, effect_add_type, effect, alpha
)